## Introduction to Quantum Computing: Labs

### Solution for Lab 7: Expectation Values

In this lab, we investigate the expectation values of observables, a central concept in quantum computing. Expectation values describe the average outcome of a measurement performed on a quantum mechanical state and allow us to determine physical quantities such as the energy of a system.

Step by step, we will implement the calculation of expectation values of arbitrary Pauli observables. We start with a single qubit and the expectation value of the Z operator. In Exercise 2, we extend this approach to an arbitrary number of qubits. In Exercise 3, we additionally include the Pauli-X and Pauli-Y operators. Finally, in Exercise 4, we generalize the procedure to linear combinations of observables, as they typically occur in variational quantum algorithms.

In [1]:
from typing import Union
import numpy as np
from qiskit import QuantumCircuit
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

### Exercise 1: The Expectation Value for 1 Qubit

In the first exercise, we write and test a function that calculates the expectation value for a single qubit with the Z operator (Pauli-Z gate).

**Exercise 1.1:** In this task, we implement a function that converts the number of measured basis states into the corresponding measured probabilities.
The function generates a Python dictionary containing the same basis states as the input dictionary.

The probabilities are calculated as follows:

$
p_i = \frac{\text{Number of measurements of bitstring } i}{\text{Total number of measurements}}
$

**Hint:** Python dictionaries can be used as follows:


In [2]:
results = {"00": 15, "01": 20, "10": 35, "11": 30}

print("Number of measurements for state 00:", results["00"])
print("Measured states:", results.keys())
print("All measurement values:", results.values())

# Loop over all measured states
for state in results.keys():
    print(f"State {state} was measured {results[state]} times.")

# Empty dictionary:
results = {}
# Add new entry:
results["00"] = 15

Number of measurements for state 00: 15
Measured states: dict_keys(['00', '01', '10', '11'])
All measurement values: dict_values([15, 20, 35, 30])
State 00 was measured 15 times.
State 01 was measured 20 times.
State 10 was measured 35 times.
State 11 was measured 30 times.


In [3]:
def measurements_to_probabilities(measurements: dict) -> dict:
    """
    Converts measurement counts to probabilities, both stored in dictionaries.

    Args:
        measurements (dict): A dictionary with measurement results as keys and their counts as values.

    Returns:
        dict: A dictionary with measurement results as keys and their probabilities as values.
    """
    probabilities = {}

    # TODO: Implementation of Exercise 1.1

    return probabilities

You can test the function with this cell here:

In [4]:
# Test for the function
measurements = {"00": 15, "01": 20, "10": 35, "11": 30}  # total of 100 measurements
probabilities = measurements_to_probabilities(measurements)
print("Probabilities:", probabilities)

Probabilities: {}


**Exercise 1.2:** In this task, we calculate the expectation value of a single Z operator (Z gate) for a quantum state of a single qubit. We write a function that receives the measurements as a Python dictionary.
Use the function from Exercise 1.1 to determine the probabilities.
The expectation value of a single Z operator can be written as:

$
\langle \Psi | Z | \Psi \rangle = p_0 - p_1
$

Note that the bitstrings for the 0 or 1 state do not necessarily appear in the measurements if they were not measured.

**Hint:** You can use the `get` function of a dictionary to specify a value that is returned if an element is not contained in the dictionary.


In [5]:
# Empty dictionary:
results = {}
# Add new entry:
results["00"] = 15
# Access an entry that does not exist (0.0 is returned)
results.get("11", 0.0)

0.0

In [6]:
def single_Z_expectation(measurements: dict) -> float:
    """
    Computes the expectation value of a single Z operator from measurement results of a single qubit state.

    Args:
        measurements (dict): A dictionary with measurement results as keys and their counts as values.

    Returns:
        float: The expectation value of the Z operator.
    """

    expectation_value = 0.0

    # TODO: Implementation of Exercise 1.2

    return expectation_value

**Exercise 1.3:** Test the function with the following measurements: `{"0": 70, "1": 30}`, `{"0": 20}` and `{"1": 15}` and print the corresponding expectation values.

In [7]:
measurements_1 = {"0": 70, "1": 30}
measurements_2 = {"0": 20}
measurements_3 = {"1": 15}

# TODO: Implementation of Exercise 1.3

### Exercise 2: Expectation Value of a Multi-Qubit Z Operator

In this task, we program a Python function with which we can calculate the expectation value of an operator consisting of multiple Z gates, for example $ Z_1 Z_2 Z_4 $.

To represent such operators compactly, we use **Pauli strings**.
A Pauli string is a string whose length corresponds to the number of qubits:

- `"Z"` means: A Z operator acts on this qubit.
- `"I"` means: Nothing happens on this qubit (identity operator).

Example for 5 qubits:

$
Z_1 Z_2 Z_4 \;\;\longrightarrow\;\; \texttt{"ZZIZI"}
$

Our Python function receives this Pauli string together with the measurement data.

For each measured bitstring result $i$ (e.g., `"10101"`), we know a probability $p_i$.
To calculate the expectation value, we must determine what contribution each individual measured bitstring makes to it.

The crucial point is: **At each position where the Pauli string has `"Z"` and the measured bitstring contains a `"1"`, a factor of $-1$ arises.**

In other words:
A Z gate flips the sign exactly when the corresponding qubit in the measurement string was in the state $|1\rangle$.

Example:

$
\langle 101 \lvert Z_1 Z_2 I_3 \rvert 101 \rangle
= \langle 1 | Z_1 | 1 \rangle \cdot \langle 0 | Z_2 | 0 \rangle \cdot \langle 1 | I_3 | 1 \rangle
= (-1)\cdot(1)\cdot(1) = -1
$

Here, there is a Z at the first qubit and the bit is `1`, hence a $-1$.
The factor $(-1)$ can also occur multiple times, e.g.:

$
\langle 11 | Z_1 Z_2 | 11 \rangle = (-1)(-1) = 1
$

For each bitstring $i$, we thus determine:

1. **the factor** $\alpha_i = \pm 1$, depending on the matches `"Z"` ↔ `"1"`,
2. **the contribution to the expectation value:** $p_i \cdot \alpha_i$.

Then we sum over all bitstrings:

$
\langle \Psi \lvert O \rvert \Psi \rangle = \sum_i \alpha_i\, p_i.
$

The algorithm can be summarized in the following pseudocode:

```
expectation_value = 0
FOR each bitstring i IN measurement_data:
    factor = 1

    FOR each position k IN pauli_string:

        IF pauli_string[k] == "Z" AND i[k] == "1" THEN
            factor = factor * (-1)

    expectation_value = expectation_value + probabilities[i] * factor

RETURN expectation_value
```


**Exercise 2.1:** Calculate the expectation value of the Z observable for multiple qubits from a given Pauli string consisting of `"I"` and `"Z"`, as well as a Python dictionary with the measured bitstrings and their counts. First determine the probabilities before summing them with the appropriate sign for the expectation value.

In [8]:
# Tip: Python allows iterating over both the indices and the values of a string:

string = "abcd"

for i, char in enumerate(string):
    print(f"Index: {i}, Character: {char}")

Index: 0, Character: a
Index: 1, Character: b
Index: 2, Character: c
Index: 3, Character: d


In [9]:
def Z_expectation_from_measurements(pauli_string: str, measurements: dict) -> float:
    """
    Implementation of the expectation value calculation for multi-qubit Z Pauli-strings.

    Args:
        pauli_string (str): A string consisting of "I" and "Z" characters representing the Pauli operator.
        measurements (dict): A dictionary with measurement results as keys and their counts as values.

    Returns:
        float: The expectation value of the given Pauli Z string operator.
    """

    probabilities = measurements_to_probabilities(measurements)

    expectation_value = 0

    # TODO Implementation of Exercise 2.1

    return expectation_value

Use the following code to test your function:

In [10]:
pauli_string = "ZIZ"
measurements = {
    "000": 8,
    "001": 6,
    "010": 2,
    "011": 4,
    "100": 5,
    "101": 2,
    "110": 4,
    "111": 1,
}
Z_expectation_from_measurements(
    pauli_string, measurements
)  # Result should be -0.1875

0

**Exercise 2.2:** In a next step, we add the possibility to the function to perform the measurements with the quantum computer itself (simulated or with a real backend).

The appropriate options to calculate the measurements and probabilities are already pre-implemented in the function.
Add the calculation of the expectation value from Exercise 2.1.

In [11]:
from qc_lecture_tools.statevector import sv_dict
from qc_lecture_tools.sampling import (
    sample_from_circuit_backend,
    sample_from_circuit,
    measurements_to_probabilities,
)

In [12]:
def Z_expectation(
    pauli_string: str,
    quantum_circuit: QuantumCircuit,
    shots: Union[None, int] = None,
    backend=None,
) -> float:
    """
    Computes the expectation value of a multi-qubit Z Pauli-string operator from a quantum circuit.

    Args:
        pauli_string (str): A string consisting of "I" and "Z" characters representing the Pauli operator.
        quantum_circuit (QuantumCircuit): The quantum circuit to be evaluated.
        shots (int or None): The number of shots for measurement. If None, statevector simulation is used.
        backend: The backend to use for execution if shots is specified.

    Returns:
        float: The expectation value of the given Pauli Z string operator.
    """

    if shots is None:
        # No shots specified, use statevector simulation to compute exact probabilities
        probabilities = sv_dict(quantum_circuit)
    else:
        # Shots specified, perform measurements
        measured_quantum_circuit = quantum_circuit.copy()
        measured_quantum_circuit.measure_all()

        if backend is None:
            # No backend specified, use local sampling from the exact simulator
            measurements = sample_from_circuit(measured_quantum_circuit, shots)
        else:
            # Backend specified, use the provided backend for sampling (fake or real hardware)
            measurements = sample_from_circuit_backend(
                measured_quantum_circuit, shots, backend
            )

        # Convert measurement counts to probabilities
        probabilities = measurements_to_probabilities(measurements)

    # Calculate the expectation value
    expectation_value = 0

    # TODO Implementation of Exercise 2.2

    return expectation_value

Use the following code to test your implementation:

In [13]:
quantum_circuit = QuantumCircuit(3)
quantum_circuit.rx(0.8, 0)
quantum_circuit.ry(0.4, 1)
quantum_circuit.cx(1, 2)

Z_expectation("ZIZ", quantum_circuit)  # Value should be approx. 0.6417093742397796

0

In [14]:
Z_expectation("ZIZ", quantum_circuit, shots=10000)  # Value should be approx. 0.65

0

In [15]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

Z_expectation(
    "ZIZ", quantum_circuit, shots=10000, backend=FakeManilaV2()
)  # Value should be approx. 0.59 - 0.63

0

In [16]:
# If your implementation of Exercise 2.2 does not work:
# from qc_lecture_tools.expectation_value import Z_expectation

### Exercise 3: Expectation Value of a General Pauli String

In this task, we extend our function so that it can calculate the expectation value of **arbitrary Pauli strings**.
Unlike in Exercise 2, where only `"Z"` and `"I"` operators occurred, `"X"` and `"Y"` gates can now also appear.

Example:

$
X_1\, Y_3\, Z_5 \quad\longrightarrow\quad \texttt{"XIYIZ"}
$

As before, each letter is assigned to a qubit.

To determine the expectation value of a Z operator, we could evaluate the measurements directly as bitstrings in the Z basis as implemented in Exercise 2.

However, this is not possible for `"X"` and `"Y"` operators:
The measurements are **always in the Z basis**! This means: We must adapt the measurement process so that the measurements effectively take place in the X or Y basis.
For us to be able to determine an expectation value for `"X"` or `"Y"`, we must **rotate** the corresponding qubit **before measurement**, so that the measurement process in the Z basis provides the same information as a true X or Y measurement.

To measure an X operator, we perform a Hadamard gate before measurement:

$
X = HZH \quad\longrightarrow\quad \langle\Psi |X|\Psi\rangle = \langle\Psi |HZH|\Psi\rangle = \langle\Psi H|Z|H\Psi\rangle
$

This means: **If there is an `"X"` in the Pauli string, we add a Hadamard gate to this qubit before measurement.**
As a result, the subsequent measurement in the Z basis corresponds to an X measurement.

For a Y measurement, we use the fact:

$
Y = SHZHS^\dagger \quad\longrightarrow\quad \langle\Psi |Y|\Psi\rangle = \langle\Psi |SHZHS^\dagger|\Psi\rangle = \langle\Psi S H|Z|HS^\dagger\Psi\rangle
$

Practically, this means: **If there is a `"Y"` in the Pauli string, we insert the gate sequence $S^\dagger$ followed by a Hadamard gate before measurement.**
This transformation rotates the qubit such that a measurement in the Z basis provides the Y information.

| in Pauli String | required quantum gates for transformation | Operator after transformation |
|-----------------|-----------------------------------------------|----------------------------------|
| `"I"`           | none                                          | `"I"`                            |
| `"Z"`           | none                                          | `"Z"`                            |
| `"X"`           | $H$                                           | `"Z"`                            |
| `"Y"`           | $S^\dagger$ followed by $H$                   | `"Z"`                            |

Below is a template for a function to compute the general expectation value.
Three things are still missing and must be implemented by you:

1. Generation of a Pauli string from `"I"` and `"Z"` that we can pass to the `Z_expectation` function. This string is stored in `Z_string`.
2. Adaptation of the circuit according to the given Pauli string with respect to the measurement basis. 
3. Final call of the `Z_expectation` function.

Implement a loop over the given Pauli string, in which the circuit is adapted (adding gates) and the new Pauli string for `Z_expectation` is created.

**Hint:** You can append the $S^\dagger$ quantum gate in Qiskit with the function `quantum_circuit.sdg(i)`.


In [17]:
def single_expectation_value(
    pauli_string: str,
    quantum_circuit: QuantumCircuit,
    shots: Union[None, int] = None,
    backend=None,
) -> float:
    """
    Computes the expectation value of a single Pauli string operator build from (X, Y, Z, I) from a quantum circuit.

    Args:
        pauli_string (str): A string consisting of "X", "Y", "Z", and "I" characters representing the Pauli operator.
        quantum_circuit (QuantumCircuit): The quantum circuit to be evaluated.
        shots (int or None): The number of shots for measurement. If None, statevector simulation is used.
        backend: The backend to use for execution if shots is specified.

    Returns:
        float: The expectation value of the given Pauli string operator.
    """

    modified_quantum_circuit = quantum_circuit.copy()

    Z_string = ""

    # TODO Implementation of Exercise 3

    return Z_expectation(Z_string, modified_quantum_circuit, shots, backend)

In [18]:
quantum_circuit = QuantumCircuit(3)
quantum_circuit.ry(0.8, 0)
quantum_circuit.ry(0.8, 1)
quantum_circuit.rz(0.3, 1)
quantum_circuit.x(1)

single_expectation_value(
    "XYZ", quantum_circuit
)  # Value should be -0.15207462776311445

0

In [19]:
single_expectation_value(
    "XYZ", quantum_circuit, shots=10000
)  # Value should be approx. -0.15

0

In [20]:
single_expectation_value(
    "XYZ", quantum_circuit, shots=10000, backend=FakeManilaV2()
)  # Value should be approx. between -0.12 and -0.15

0

### Exercise 4: General Observables

In quantum computing, linear combinations of individual operators (Pauli strings) often appear, for example: $O = 0.25 X_1 Y_2Z_3 + 0.3 X_2Z_3$.

Here, too, we can separate the terms:

$
\braket{\Psi| 0.25 X_1 Y_2Z_3 + 0.3 X_2Z_3|\Psi} = 0.25 \braket{\Psi|X_1 Y_2Z_3|\Psi} + 0.3 \braket{\Psi|X_2Z_3|\Psi}
$

We have already implemented the individual terms on the right in Exercise 3. So only splitting and summing is missing.

For general operators, we use the data structure `SparsePauliOp` in Qiskit as above, which can represent these combinations.

Here we can initialize the individual Pauli strings and write them together as a sum:

```python
observable = 0.25 * SparsePauliOp("XYZ") + 0.3 * SparsePauliOp("IXZ")
```

In [21]:
from qiskit.quantum_info import SparsePauliOp

# Examples for the SparsePauliOp class

# Initialization of an observable consisting of two Pauli strings
observable = 0.25 * SparsePauliOp("XYZ") + 0.3 * SparsePauliOp("IXZ")
print(observable)

# Access to the individual Pauli strings and their coefficients
for op in observable:
    print(str(op.paulis[0]), op.coeffs[0])

SparsePauliOp(['XYZ', 'IXZ'],
              coeffs=[0.25+0.j, 0.3 +0.j])
XYZ (0.25+0j)
IXZ (0.3+0j)


Now implement the function that sums over the individual terms, calling the ``single_expectation_value`` function and returning the total expectation value.

In [22]:
def expectation_value(
    observable: SparsePauliOp,
    quantum_circuit: QuantumCircuit,
    shots: Union[None, int] = None,
    backend=None,
) -> float:
    """
    Computes the expectation value of a given SparsePauliOp observable from a quantum circuit.

    Args:
        observable (SparsePauliOp): The observable to be evaluated.
        quantum_circuit (QuantumCircuit): The quantum circuit to be evaluated.
        shots (int or None): The number of shots for measurement. If None, statevector simulation is used.
        backend: The backend to use for execution if shots is specified.

    Returns:
        float: The expectation value of the given observable.
    """

    expectation_value = 0.0

    # TODO Implementation of Exercise 4

    return float(np.real_if_close(expectation_value))

You can test your implementation again with the following code:

In [23]:
observable = 0.5 * SparsePauliOp("YX") + 0.25 * SparsePauliOp("XY")
quantum_circuit = QuantumCircuit(2)
quantum_circuit.ry(0.8, 0)
quantum_circuit.rz(0.8, 0)
quantum_circuit.ry(0.9, 1)
quantum_circuit.rz(0.9, 1)

expectation_value(observable, quantum_circuit)  # Should be approx. 0.20195286

0.0

In [24]:
expectation_value(observable, quantum_circuit, shots=10000)  # Should be approx. 0.20195286

0.0

In [25]:
expectation_value(observable, quantum_circuit, shots=10000, backend=FakeManilaV2())

0.0

### Exercise 5: Standard Deviation

Now that we can generally calculate the expectation value of arbitrary Pauli observables, we can next calculate the standard deviation.
The standard deviation describes the dispersion of the expectation value given a finite number of measurements.

We can calculate the standard deviation for an arbitrary observable $O$ as follows:

$
\mathrm{std}(O) = \sqrt{ \left( \braket{\Psi|OO|\Psi} - \braket{\Psi|O|\Psi}^2 \right   ) / N_\text{shots} }
$

The square $OO = O^2$ of an observable corresponds to the consecutive execution of the operators, i.e.,
$(Z_1 X_2)^2 = (Z_1 X_2)(Z_1 X_2) = (Z_1 Z_1)(X_2 X_2) = I \otimes I$.

For multiple terms, one has to multiply out again, whereby many terms usually cancel each other out. We can have Qiskit do this automatically.


In [26]:
observable = 0.5 * SparsePauliOp("YX") + 0.25 * SparsePauliOp("ZZ")

# This call squares the observable and simplifies the result
observable_squared = observable.power(2).simplify()

print(observable_squared)

SparsePauliOp(['II', 'XY'],
              coeffs=[0.3125+0.j, 0.25  +0.j])


Now implement the function that calculates the standard deviation according to the formula above.
To do this, subtract the squared expectation value $\braket{\Psi|O|\Psi}^2$ from the expectation value of the squared observable $\braket{\Psi|OO|\Psi}$.
Calculate these two values first **exactly** without shots (`shots=None`) and then use the passed shots to apply the formula for the standard deviation.

**Hint:** You can use the functions `np.sqrt()` and `np.square()` for the square root and squaring.

In [27]:
def standard_deviation(
    observable: SparsePauliOp, quantum_circuit: QuantumCircuit, shots: int
):
    """
    Computes the standard deviation of a given SparsePauliOp observable from a quantum circuit.

    Args:
        observable (SparsePauliOp): The observable to be evaluated.
        quantum_circuit (QuantumCircuit): The quantum circuit to be evaluated.
        shots (int): The number of shots for computing the standard deviation.

    Returns:
        float: The standard deviation of the given observable.
    """


    std = 0.0

    # TODO Implementation of Exercise 5

    return std

Use the following code to test your implementation

In [28]:
observable = 0.5 * SparsePauliOp("YX") + 0.25 * SparsePauliOp("XY")
quantum_circuit = QuantumCircuit(2)
quantum_circuit.ry(0.8, 0)
quantum_circuit.rz(0.8, 0)
quantum_circuit.ry(0.9, 1)
quantum_circuit.rz(0.9, 1)

standard_deviation(observable, quantum_circuit, 1000)  # Should be approx. 0.01949320

0.0

### Exercise 6: Analysis

Finally, we analyze the finite sampling noise of the expectation value.
For this, we determine the expectation value from a finite number of measurements.
Since the measurements are random, this result always contains a certain random component, whose influence we examine more closely here. For the experiments, we use a fixed number of 1000 measurements (`shots=1000`).

First, we calculate the theoretical expectation value (`shots=None`) and its standard deviation and output them.
Then we calculate the expectation value with 1000 measurements.
We repeat the expectation value calculation 500 times to subsequently be able to analyze a distribution of the obtained expectation values.
Plotting and evaluation are already implemented, so you only need to add the theoretical values and the expectation value calculation with a finite number of measurements.

Save the individual expectation values in the list `erwartungswerte` (`erwartungswerte.append(value)`).


In [29]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Helper function for the analysis of the expectation values
# This function is not part of the exercises, but can be used to analyze the distribution of expectation values obtained from measurements.

def analyse(expectation_values: list):
    """
    Histogram and Gaussian distribution fit of the expectation values.

    Args:
        expectation_values (list): List of calculated expectation values

    Returns:
        Graphic with the analysis
    """

    # Convert to the correct format for analysis
    expectation_values = np.array(expectation_values).reshape(-1, 1)

    # Fit a Gaussian distribution to the data
    mu, sigma = norm.fit(expectation_values)

    # Create histogram
    plt.figure(figsize=(7, 4))
    count, bins, ignored = plt.hist(
        expectation_values, bins=40, density=True, alpha=0.6, edgecolor="black"
    )

    # Create x-values for the fitted curve
    x = np.linspace(min(expectation_values), max(expectation_values), 1000)
    pdf = norm.pdf(x, mu, sigma)

    # Plot the fitted Gaussian curve
    plt.plot(x, pdf, "r-", linewidth=2, label=f"Fit: μ={mu:.3f}, σ={sigma:.3f}")

    plt.title("Histogram of the expectation value with fitted Gaussian curve")
    plt.xlabel(r"Expectation Value")
    plt.ylabel("Frequency of measured exp. value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    return plt

In [30]:
# Circuit for the analysis (can also be modified if you like)

quantum_circuit = QuantumCircuit(3)
quantum_circuit.ry(0.8, 0)
quantum_circuit.rz(0.8, 0)
quantum_circuit.ry(0.9, 1)
quantum_circuit.rz(0.9, 1)
quantum_circuit.ry(2.0, 1)
quantum_circuit.rz(3.0, 1)

quantum_circuit.cx(0, 1)
quantum_circuit.cx(1, 2)
quantum_circuit.cx(2, 0)

observable = 0.5 * SparsePauliOp("YXZ") + 0.25 * SparsePauliOp("ZXY")

**Exercise 6.1:** Analyze the expectation value for the given circuit and the given observable. We use the ideal simulation here, but with a limited number of 1000 measurements.

In [31]:
# TODO Implementation Exercise 6.1

**Exercise 6.2:** Now we repeat the experiment with a simulated hardware backend (`FakeManilaV2()`) to perform the repeated measurements.
This will likely compute for a few minutes. (Theoretically, you can also use a real backend, but it will cost quite a bit of computation time.)

Compare the mean value found (marked with µ in the plot) with the theoretical value and the result from Exercise 6.1. What do you notice?

In [32]:
# TODO Implementation Exercise 6.2